In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [4]:
# run search
result = tavily_client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]

'The new Blackwell GPU by Nvidia features a custom-built TSMC 4NP process and 208 billion transistors, designed for AI and cloud computing workloads.'

Regular search

In [5]:
city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

In [6]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"DDG 요청 중 예외 발생으로 이전 결과를 반환합니다.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

/var/folders/cm/n6cjvrh90t5gfdsjln2hzgc40000gn/T/ipykernel_26933/516628885.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


https://weather.com/en-NA/weather/today/l/San+Francisco+CA+United+States?canonicalCityId=d060bcef6a904af38313393bd51e4c3c
https://www.accuweather.com/en/us/san-francisco/94103/hourly-weather-forecast/347629
https://www.theweathernetwork.com/en/city/us/california/san-francisco/current
https://www.weather-forecast.com/locations/San-Francisco/forecasts/latest
https://www.weather-us.com/en/california-usa/san-francisco
https://www.localconditions.com/us/san-francisco/california/weather/


In [7]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [8]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

Website: https://weather.com/en-NA/weather/today/l/San+Francisco+CA+United+States?canonicalCityId=d060bcef6a904af38313393bd51e4c3c


<body><div class="appWrapper DaybreakLargeScreen LargeScreen lightTheme twcTheme DaybreakLargeScreen--appWrapper--ZkDop" id="appWrapper"><div class="region-sidebarNavigation regionSidebarNavigation"><div class="removeIfEmpty" id="WxuSidebarNavigation-sidebarNavigation-sidebar-nav-injection"><div aria-hidden="false" class="SidebarNavigation--sidebarWrapper--v+g78"><nav aria-label="Site Navigation Links" class="SidebarNavigation--sidebar--A5yHr" data-testid="sidebar-navigation"><div class="SidebarNavigation--sidebarHeader--bMAxo"><button class="SidebarNavigation--overlayControlButton--cSFvn openSidebarOverlayButton" tabindex="0" type="button"><span class="Icon--iconWrapper--vSeDL"><svg class="Icon--icon--ySD-o" fill="none" height="24" name="hamburger" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg">
<title>Hamburger</title>
<path d="M20 16

In [9]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/en-NA/weather/today/l/San+Francisco+CA+United+States?canonicalCityId=d060bcef6a904af38313393bd51e4c3c


Recents Products & Account Special Forecasts Rain Rain ending around 22:30. Radar 05:47 20:30 Morning Afternoon Evening Overnight Hourly Forecast Now 23:00 00:00 01:00 02:00 Daily Forecast Today Tue 09 Wed 10 Thu 11 Fri 12 Air Quality Index Air quality is considered satisfactory, and air pollution poses little or no risk. Health & Activities Seasonal Allergies and Pollen Count Forecast Grass pollen is moderate in your area We recognise our responsibility to use data and technology for good. We may use or share your data with our data vendors. Take control of your data. The Weather Channel is the world's most accurate forecaster according to ForecastWatch, Global and Regional Weather Forecast Accuracy Overview , 2021-2024, commissioned by The Weather Company . Weather Channel © The Weather Company, LLC 2026


Agentic Search

In [10]:
# run search
result = tavily_client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1780981974, 'localtime': '2026-06-08 22:12'}, 'current': {'last_updated_epoch': 1780981200, 'last_updated': '2026-06-08 22:00', 'temp_c': 15.0, 'temp_f': 59.0, 'is_day': 0, 'condition': {'text': 'Overcast', 'icon': '//cdn.weatherapi.com/weather/64x64/night/122.png', 'code': 1009}, 'wind_mph': 4.9, 'wind_kph': 7.9, 'wind_degree': 239, 'wind_dir': 'WSW', 'pressure_mb': 1016.0, 'pressure_in': 30.01, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 90, 'cloud': 100, 'feelslike_c': 14.9, 'feelslike_f': 58.8, 'windchill_c': 12.3, 'windchill_f': 54.1, 'heatindex_c': 13.2, 'heatindex_f': 55.8, 'dewpoint_c': 13.0, 'dewpoint_f': 55.4, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 7.9, 'gust_kph': 12.7, 'will_it_rain': 0, 'chance_of_rain': 54, 'will_it_snow': 0, 'chance_of_snow': 0}}


In [11]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)

{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1780981974,
        "localtime": "2026-06-08 22:12"
    },
    "current": {
        "last_updated_epoch": 1780981200,
        "last_updated": "2026-06-08 22:00",
        "temp_c": 15.0,
        "temp_f": 59.0,
        "is_day": 0,
        "condition": {
            "text": "Overcast",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/122.png",
            "code": 1009
        },
        "wind_mph": 4.9,
        "wind_kph": 7.9,
        "wind_degree": 239,
        "wind_dir": "WSW",
        "pressure_mb": 1016.0,
        "pressure_in": 30.01,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 90,
        "cloud": 100,
        "feelslike_c": 14.9,
        "feelslike_f": 58.8,
        "windchill_c": 12.3,
       